# Reconstruction Analysis (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
if 'google.colab' in sys.modules:
    try:
        import pyvis  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyvis'])

## Step 1: Load Train/Val/Test Splits
Load correlation matrices from an existing dataset folder containing `train.pt`, `val.pt`, and `test.pt`.
- Local PC: `data/processed/dataset/<DATASET_NAME>`
- Google Colab: `dataset_tesi/<DATASET_NAME>`

In [ ]:
DATASET_NAME = 'data_00_20_w724_s10'

if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / 'dataset'

dataset_dir = base_dir / DATASET_NAME

TRAIN_FILE = dataset_dir / 'train.pt'
VAL_FILE = dataset_dir / 'val.pt'
TEST_FILE = dataset_dir / 'test.pt'

for split_path in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    if not split_path.exists():
        raise FileNotFoundError(
            f"File '{split_path.name}' not found in: {dataset_dir.absolute()}"
        )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Train: {TRAIN_FILE.name} | Val: {VAL_FILE.name} | Test: {TEST_FILE.name}')

In [ ]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), meta


train_corr, train_meta = load_corr_payload(TRAIN_FILE)
val_corr, val_meta = load_corr_payload(VAL_FILE)
test_corr, test_meta = load_corr_payload(TEST_FILE)

if train_corr.shape[1:] != test_corr.shape[1:]:
    raise ValueError(f'Train/test asset dims mismatch: {train_corr.shape} vs {test_corr.shape}')
if val_corr.shape[1:] != train_corr.shape[1:]:
    raise ValueError(f'Val/train asset dims mismatch: {val_corr.shape} vs {train_corr.shape}')

print(f'train_corr shape: {tuple(train_corr.shape)}')
print(f'val_corr shape:   {tuple(val_corr.shape)}')
print(f'test_corr shape:  {tuple(test_corr.shape)}')

## Step 2: Prepare Matrices from Existing Splits
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.
Train/val/test come directly from the loaded split files (no random split in notebook).

In [ ]:
train_np = train_corr.numpy().astype(np.float32)
val_np = val_corr.numpy().astype(np.float32)
test_np = test_corr.numpy().astype(np.float32)
all_np = np.concatenate([train_np, val_np, test_np], axis=0)

n_train, n_assets, _ = train_np.shape
n_val = val_np.shape[0]
n_test = test_np.shape[0]
n_all = all_np.shape[0]
n_matrices = n_all
n_features = n_assets * n_assets

x_train = torch.from_numpy(train_np.reshape(n_train, n_features))
x_val = torch.from_numpy(val_np.reshape(n_val, n_features))
x_test = torch.from_numpy(test_np.reshape(n_test, n_features))
x_all = torch.from_numpy(all_np.reshape(n_all, n_features))

VAL_FRACTION = n_val / n_matrices
TEST_FRACTION = n_test / n_matrices

print(f'Number of matrices: {n_matrices}')
print(f'Matrix shape: ({n_assets}, {n_assets})')
print(f'Flattened input size: {n_features}')
print(f'Train size: {n_train} | Val size: {n_val} | Test size: {n_test}')
print(f'All size: {n_all}')

## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Linear(latent_dim, input_dim)

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]

        dimensions = [input_dim, *hidden_dims, latent_dim]

        encoder_layers = []
        for i in range(len(dimensions) - 1):
            encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
            if i < len(dimensions) - 2:
                encoder_layers.append(nn.ReLU())
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_dims = dimensions[::-1]
        decoder_layers = []
        for i in range(len(decoder_dims) - 1):
            decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
            if i < len(decoder_dims) - 2:
                decoder_layers.append(nn.ReLU())
            else:
                decoder_layers.append(nn.Tanh())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [ ]:
RESULTS_JSON_PATH = 'best_models_tesi/linearAE/linearAE_005/linear_AE_results_linearAE_005.json'
MODEL_TYPE = None  # Set to 'linear' or 'ae' to override detection
WEIGHTS_OVERRIDE_PATH = None  # Optional: set a .pt path if JSON does not include it


def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p


results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

model_cfg = results.get('model', {})
latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
if MODEL_TYPE is None:
    model_type = 'ae' if hidden_dims is not None else 'linear'
else:
    model_type = str(MODEL_TYPE).strip().lower()

if model_type not in {'ae', 'linear'}:
    raise ValueError("MODEL_TYPE must be 'linear' or 'ae'")
if model_type == 'ae' and hidden_dims is None:
    raise ValueError('hidden_dims missing in results JSON for AE model')

input_dim = int(model_cfg.get('input_dim', x_train.shape[1]))
if input_dim != x_train.shape[1]:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={x_train.shape[1]}')

if model_type == 'linear':
    model = LinearAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
else:
    model = AutoEncoder(input_dim=input_dim, latent_dim=latent_dim, hidden_dims=hidden_dims).to(device)

weights_path = WEIGHTS_OVERRIDE_PATH
if weights_path is None:
    weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    if model_type == 'linear':
        weights_path = f'linear_AE_best_{run_name}.pt'
    else:
        weights_path = f'AE_best_{run_name}.pt'

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)
if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

print(f'Model type: {model_type} | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')

## Step 5: Latent Space Analysis
Encode the matrices into the latent space and analyze feature distributions.

In [ ]:
def compute_latents(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 256):
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encoder(xb)
            latents.append(z.cpu().numpy())

    return np.concatenate(latents, axis=0)


latents_all = compute_latents(model, x_all, batch_size=256)
latent_dim = latents_all.shape[1]
latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
latent_df = pd.DataFrame(latents_all, columns=latent_cols)

print('Latent distribution summary (all data):')
display(latent_df.describe().T)

valid_cols = [col for col in latent_cols if latent_df[col].notna().any() and latent_df[col].nunique() > 1]
latent_df = latent_df[valid_cols]

analysis_dir = results_path.parent / 'analysis_outputs'
analysis_dir.mkdir(parents=True, exist_ok=True)
latent_json_path = analysis_dir / 'latent_all.json'
latent_df.to_json(latent_json_path, orient='records', indent=2)
print(f'Saved latent samples: {latent_json_path}')

if len(valid_cols) < 2:
    print('Not enough valid latent dimensions for pairwise plots.')
else:
    grid = sns.PairGrid(latent_df, corner=True, diag_sharey=False)
    grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
    grid.fig.suptitle('Pairwise latent dimension plots (all data)', y=1.02)
    pairplot_path = analysis_dir / 'latent_pairwise_all.png'
    grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved latent pairwise plot: {pairplot_path}')

## Step 6: Reconstruction Performance (All Data)
Reconstruct matrices and compute MSE, MAE, and Frobenius norm on the full dataset (train + val + test).

In [ ]:
def sanitize_reconstruction(matrix: np.ndarray) -> np.ndarray:
    """Normalize a reconstructed correlation matrix to valid form."""
    matrix = np.asarray(matrix)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f'Expected a square 2D matrix, got shape {matrix.shape}')

    sym = 0.5 * (matrix + matrix.T)
    sanitized = sym.copy()

    n = sanitized.shape[0]
    off_mask = ~np.eye(n, dtype=bool)
    sanitized[off_mask] = np.clip(sanitized[off_mask], -1.0, 1.0)

    diag = np.diag(sanitized)
    denom = np.sqrt(np.outer(diag, diag))
    sanitized = sanitized / denom
    np.fill_diagonal(sanitized, 1.0)

    return sanitized


def reconstruct_matrices(model: nn.Module, x_tensor: torch.Tensor, n_assets: int, batch_size: int = 64):
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    recon = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            x_hat = model(xb)
            recon.append(x_hat.cpu().numpy())

    recon_flat = np.concatenate(recon, axis=0)
    return recon_flat.reshape(-1, n_assets, n_assets)


def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    if original.shape != reconstructed.shape:
        raise ValueError(f'Shape mismatch: {original.shape} vs {reconstructed.shape}')

    diff = original - reconstructed
    mse = np.mean(diff ** 2, axis=(1, 2))
    mae = np.mean(np.abs(diff), axis=(1, 2))
    frob = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    errors_df = pd.DataFrame({'mse': mse, 'mae': mae, 'frobenius': frob})
    summary_df = (
        errors_df.agg(['mean', 'median', 'std', 'min', 'max'])
        .T.reset_index()
        .rename(columns={'index': 'metric'})
    )
    return errors_df, summary_df


def evaluate_split(split_name: str, original_np: np.ndarray, x_tensor: torch.Tensor):
    recon_corr = reconstruct_matrices(model, x_tensor, n_assets=n_assets, batch_size=64)
    errors_df, summary_df = reconstruction_errors(original_np, recon_corr)
    errors_df.insert(0, 'split', split_name)
    summary_df.insert(0, 'split', split_name)

    print(f'Reconstruction error summary ({split_name} set):')
    display(summary_df)
    return errors_df, summary_df

In [ ]:
analysis_dir = results_path.parent / 'analysis_outputs'
analysis_dir.mkdir(parents=True, exist_ok=True)

recon_all = reconstruct_matrices(model, x_all, n_assets=n_assets, batch_size=64)
errors_df, summary_df = reconstruction_errors(all_np, recon_all)

print('Reconstruction error summary (all data):')
display(summary_df)

errors_json_path = analysis_dir / 'reconstruction_errors_all.json'
summary_json_path = analysis_dir / 'reconstruction_summary_all.json'
errors_df.to_json(errors_json_path, orient='records', indent=2)
summary_df.to_json(summary_json_path, orient='records', indent=2)

print(f'Saved per-matrix errors: {errors_json_path}')
print(f'Saved summary stats: {summary_json_path}')

## Step 7: MST Reconstruction Analysis (All Data)
Compare the MST of original vs reconstructed (sanitized) correlation matrices on the full dataset (train + val + test) and aggregate metrics across all matrices.
NetworkX plots are color-coded by GICS sector using the same palette as notebook 03 (from `../results/gics_by_ticker.csv`).

In [ ]:
import networkx as nx

MST_SAMPLE_SPLIT = 'all'  # 'all' = train + val + test
MST_SAMPLE_INDEX = 0
MAX_MATRICES_PER_SPLIT = None  # Set an int to limit runtime on all data
BETWEENNESS_TOP_K = 10
RUN_SAMPLE_MST = True
RUN_AGGREGATE_MST = True

analysis_dir = results_path.parent / 'analysis_outputs'
analysis_dir.mkdir(parents=True, exist_ok=True)
mst_dir = analysis_dir / 'mst_reconstruction'
mst_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def sanitize_original_matrix(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f'Expected a square 2D matrix, got shape {matrix.shape}')
    matrix = np.clip(matrix, -1.0, 1.0)
    matrix = 0.5 * (matrix + matrix.T)
    np.fill_diagonal(matrix, 1.0)
    return matrix


def build_distance_matrix(corr_matrix: np.ndarray) -> np.ndarray:
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = 0.5 * (dist_matrix + dist_matrix.T)
    np.fill_diagonal(dist_matrix, 0.0)
    return dist_matrix


def build_mst_from_corr(corr_matrix: np.ndarray):
    dist_matrix = build_distance_matrix(corr_matrix)
    graph = nx.from_numpy_array(dist_matrix)
    mst = nx.minimum_spanning_tree(graph, weight='weight')
    return dist_matrix, mst


def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}


def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

In [ ]:
def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [ ]:
def get_split_data(split_name: str):
    split = str(split_name).strip().lower()
    if split == 'all':
        base_meta = None
        for meta in (train_meta, val_meta, test_meta):
            if isinstance(meta, dict):
                base_meta = meta
                break
        return all_np, x_all, base_meta
    if split == 'train':
        return train_np, x_train, train_meta
    if split == 'val':
        return val_np, x_val, val_meta
    if split == 'test':
        return test_np, x_test, test_meta
    raise ValueError("split_name must be 'all', 'train', 'val', or 'test'")


def get_tickers(meta, n_assets: int, fallback_tickers=None):
    tickers = None
    if isinstance(meta, dict):
        tickers = meta.get('tickers')
    if tickers is not None:
        tickers = list(tickers)
        if len(tickers) == n_assets:
            return [str(t) for t in tickers]
    if fallback_tickers is not None:
        tickers = list(fallback_tickers)
        if len(tickers) == n_assets:
            return [str(t) for t in tickers]
    return [f'A{i}' for i in range(n_assets)]


def load_gics_map():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        gics_path = DRIVE_ROOT / 'dataset_tesi' / 'gics_by_ticker.csv'
    else:
        gics_path = Path('../results/gics_by_ticker.csv')

    if not gics_path.exists():
        raise FileNotFoundError(f'Missing GICS CSV: {gics_path}')
    gics_df = pd.read_csv(gics_path)
    gics_df['ticker'] = gics_df['ticker'].astype(str).str.upper()
    gics_df['gics_category'] = gics_df['gics_category'].astype(str)
    gics_map = dict(zip(gics_df['ticker'], gics_df['gics_category']))
    gics_tickers = gics_df['ticker'].tolist()
    return gics_map, gics_tickers

In [ ]:
def plot_mst_graph(mst, tickers, out_png, out_html, title, gics_map):
    from matplotlib.patches import Patch

    node_labels = {i: str(tickers[i]) for i in range(len(tickers))}
    mst_labeled = nx.relabel_nodes(mst, node_labels)

    degree_dict = dict(mst_labeled.degree())
    node_names = list(mst_labeled.nodes())
    node_degrees = np.array([degree_dict[n] for n in node_names], dtype=float)
    node_sectors = [gics_map.get(str(n).upper(), 'Unknown') for n in node_names]

    sector_order = [
        'Energy',
        'Materials',
        'Industrials',
        'Consumer Discretionary',
        'Consumer Staples',
        'Health Care',
        'Financials',
        'Information Technology',
        'Communication Services',
        'Utilities',
        'Real Estate',
        'Unknown',
    ]
    sector_colors = [
        '#1f77b4',
        '#ff7f0e',
        '#2ca02c',
        '#d62728',
        '#9467bd',
        '#8c564b',
        '#e377c2',
        '#7f7f7f',
        '#bcbd22',
        '#17becf',
        '#aec7e8',
        '#ffffff',
    ]
    color_map = {sector: sector_colors[i] for i, sector in enumerate(sector_order)}
    node_colors = [color_map.get(sector, '#9e9e9e') for sector in node_sectors]

    def radial_tree_layout(
        tree,
        root=None,
        min_radius=0.35,
        max_radius=1.0,
        radial_power=1.35,
        angle_gap=0.035,
    ):
        if len(tree) == 0:
            return {}, None
        if root is None:
            root = max(tree.degree, key=lambda x: x[1])[0]
        bfs_tree = nx.bfs_tree(tree, root)
        children = {n: list(bfs_tree.successors(n)) for n in bfs_tree.nodes()}
        leaf_counts = {}

        def compute_leaf_counts(node):
            kids = children.get(node, [])
            if not kids:
                leaf_counts[node] = 1
                return 1
            total = 0
            for child in kids:
                total += compute_leaf_counts(child)
            leaf_counts[node] = total
            return total

        compute_leaf_counts(root)
        depths = nx.single_source_shortest_path_length(bfs_tree, root)
        max_depth = max(depths.values()) if depths else 0
        pos = {}

        def assign_angles(node, angle_start, angle_end):
            angle = 0.5 * (angle_start + angle_end)
            depth = depths.get(node, 0)
            if max_depth <= 0:
                radius = 0.0
            else:
                t = depth / max_depth
                radius = min_radius + (max_radius - min_radius) * (t ** radial_power)
            pos[node] = np.array([radius * np.cos(angle), radius * np.sin(angle)])

            kids = children.get(node, [])
            if not kids:
                return
            kids = sorted(kids, key=lambda n: (-leaf_counts.get(n, 1), str(n)))
            span = angle_end - angle_start
            gap = max(0.0, angle_gap)
            total_gap = gap * max(0, len(kids) - 1)
            available_span = max(1e-6, span - total_gap)
            total = sum(leaf_counts.get(k, 1) for k in kids)
            current = angle_start
            for child in kids:
                frac = leaf_counts.get(child, 1) / total if total > 0 else 1.0 / len(kids)
                child_span = available_span * frac
                assign_angles(child, current, current + child_span)
                current += child_span + gap

        assign_angles(root, 0.0, 2.0 * np.pi)
        return pos, root

    layout_used = 'radial'
    try:
        from networkx.drawing.nx_agraph import graphviz_layout
        root = max(mst_labeled.degree, key=lambda x: x[1])[0]
        pos = graphviz_layout(
            mst_labeled,
            prog='twopi',
            root=root,
            args='-Goverlap=false -Gsep=+0.8 -Granksep=1.1 -Gnodesep=0.4',
        )
        layout_used = 'graphviz-twopi'
    except Exception:
        pos, root = radial_tree_layout(mst_labeled)

    pos_keys = list(pos.keys())
    pos_arr = np.array([pos[k] for k in pos_keys], dtype=float)
    mins = pos_arr.min(axis=0)
    maxs = pos_arr.max(axis=0)
    span = np.maximum(maxs - mins, 1e-12)
    pos_arr = (pos_arr - mins) / span
    pos_arr = pos_arr * 1.9 - 0.95
    pos = {k: pos_arr[idx] for idx, k in enumerate(pos_keys)}

    fig, ax = plt.subplots(figsize=(26, 20))
    nx.draw_networkx_edges(
        mst_labeled,
        pos,
        width=1.1,
        alpha=0.72,
        edge_color='gray',
        ax=ax,
    )
    nx.draw_networkx_nodes(
        mst_labeled,
        pos,
        nodelist=node_names,
        node_size=200 + 90 * node_degrees,
        node_color=node_colors,
        edgecolors='black',
        linewidths=0.7,
        ax=ax,
    )
    nx.draw_networkx_labels(
        mst_labeled,
        pos,
        font_size=7,
        font_weight='bold',
        font_color='black',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=0.2),
        ax=ax,
    )

    unique_sectors = [s for s in sector_order if s in set(node_sectors)]
    if 'Unknown' not in unique_sectors:
        unique_sectors.append('Unknown')
    legend_handles = [Patch(facecolor=color_map[s], edgecolor='black', label=s) for s in unique_sectors]
    ax.legend(handles=legend_handles, title='GICS Sector', loc='upper left', fontsize=9, title_fontsize=10)

    ax.set_title(f'{title}\nLayout: {layout_used}', fontsize=12, fontweight='bold')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(out_png, dpi=220, bbox_inches='tight')
    print(f'Saved MST plot: {out_png}')
    plt.show()

    try:
        from pyvis.network import Network
        net = Network(height='900px', width='100%', bgcolor='#ffffff', font_color='black')
        net.barnes_hut(gravity=-20000, central_gravity=0.25, spring_length=220, spring_strength=0.01, damping=0.9)
        pos_scale = 1000.0
        for name, degree, sector in zip(node_names, node_degrees, node_sectors):
            x, y = pos.get(name, (0.0, 0.0))
            net.add_node(
                name,
                label=str(name),
                title=f'{name} | {sector} | degree={int(degree)}',
                color=color_map.get(sector, '#9e9e9e'),
                size=10 + 6 * degree,
                x=float(x * pos_scale),
                y=float(y * pos_scale),
            )
        for u, v in mst_labeled.edges():
            net.add_edge(u, v, color='#999999')

        net.save_graph(str(out_html))
        print(f'Saved MST PyVis: {out_html}')
    except Exception as exc:
        print(f'PyVis not available or failed to render: {exc}')


def plot_degree_distribution(deg_orig, deg_recon, out_png):
    max_degree = int(max(deg_orig.max(), deg_recon.max()))
    counts_orig = np.bincount(deg_orig, minlength=max_degree + 1)
    counts_recon = np.bincount(deg_recon, minlength=max_degree + 1)
    freq_orig = counts_orig / counts_orig.sum()
    freq_recon = counts_recon / counts_recon.sum()

    degrees = np.arange(max_degree + 1)
    valid = (freq_orig + freq_recon) > 0
    degrees = degrees[valid]
    freq_orig = freq_orig[valid]
    freq_recon = freq_recon[valid]

    deg_mask = degrees > 0
    degrees = degrees[deg_mask]
    pct_orig = freq_orig[deg_mask] * 100.0
    pct_recon = freq_recon[deg_mask] * 100.0

    value_mask = (pct_orig > 0) | (pct_recon > 0)
    degrees = degrees[value_mask]
    pct_orig = pct_orig[value_mask]
    pct_recon = pct_recon[value_mask]

    if len(degrees) == 0:
        print('No positive degree frequencies to plot in log-log scale.')
        return

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(degrees, pct_orig, s=45, alpha=0.8, color='steelblue', edgecolor='black', label='Original')
    ax.scatter(degrees, pct_recon, s=45, alpha=0.8, color='crimson', edgecolor='black', label='Reconstructed')

    if len(degrees) >= 2:
        slope_orig, intercept_orig = np.polyfit(np.log10(degrees), np.log10(pct_orig), 1)
        y_fit_orig = 10 ** (intercept_orig + slope_orig * np.log10(degrees))
        ax.plot(degrees, y_fit_orig, color='steelblue', linewidth=2, linestyle='--',
                label=f'Original fit slope={slope_orig:.3f}')

        slope_recon, intercept_recon = np.polyfit(np.log10(degrees), np.log10(pct_recon), 1)
        y_fit_recon = 10 ** (intercept_recon + slope_recon * np.log10(degrees))
        ax.plot(degrees, y_fit_recon, color='crimson', linewidth=2, linestyle='--',
                label=f'Reconstructed fit slope={slope_recon:.3f}')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Degree (k)')
    ax.set_ylabel('Nodes with degree k (%)')
    ax.set_title('MST degree distribution (log-log)')
    ax.grid(True, which='both', alpha=0.3)

    xticks = np.unique(degrees.astype(int))
    xticks = xticks[xticks > 0]
    if len(xticks) > 0:
        ax.set_xticks(xticks)
        ax.set_xticklabels([f'{int(k)}' for k in xticks], rotation=45, ha='right')

    yticks = np.unique(np.concatenate([pct_orig, pct_recon]))
    yticks = yticks[yticks > 0]
    if len(yticks) > 0:
        ax.set_yticks(yticks)
        ax.set_yticklabels([f'{val:.4f}%' for val in yticks])

    ax.legend()
    fig.tight_layout()
    fig.savefig(out_png, dpi=160, bbox_inches='tight')
    print(f'Saved degree distribution: {out_png}')
    plt.show()


def run_sample_mst_comparison(split_name: str, matrix_idx: int):
    original_np, x_tensor, meta = get_split_data(split_name)
    if not (0 <= matrix_idx < original_np.shape[0]):
        raise ValueError(f'Invalid matrix index {matrix_idx} for split {split_name}')

    recon_matrix = reconstruct_matrices(model, x_tensor[matrix_idx:matrix_idx + 1], n_assets=n_assets, batch_size=1)[0]
    orig_corr = sanitize_original_matrix(original_np[matrix_idx])
    recon_corr = sanitize_reconstruction(recon_matrix)

    _, mst_orig = build_mst_from_corr(orig_corr)
    _, mst_recon = build_mst_from_corr(recon_corr)

    metrics = compare_mst_metrics(mst_orig, mst_recon, n_assets, BETWEENNESS_TOP_K)
    metrics.update({'split': split_name, 'matrix_idx': int(matrix_idx)})

    sample_dir = mst_dir / f'sample_{split_name}_{matrix_idx}'
    sample_dir.mkdir(parents=True, exist_ok=True)

    gics_map, gics_tickers = load_gics_map()
    tickers = get_tickers(meta, n_assets, fallback_tickers=gics_tickers)
    matched = sum(1 for t in tickers if str(t).upper() in gics_map)
    if matched == 0:
        print('Warning: no tickers matched GICS map; nodes will be colored as Unknown.')
    elif matched < len(tickers):
        print(f'Warning: {matched}/{len(tickers)} tickers matched GICS map; unmatched will be Unknown.')

    plot_mst_graph(
        mst_orig,
        tickers,
        out_png=sample_dir / 'mst_original_networkx.png',
        out_html=sample_dir / 'mst_original_pyvis.html',
        title=f'MST Original ({split_name} idx={matrix_idx})',
        gics_map=gics_map,
    )
    plot_mst_graph(
        mst_recon,
        tickers,
        out_png=sample_dir / 'mst_reconstructed_networkx.png',
        out_html=sample_dir / 'mst_reconstructed_pyvis.html',
        title=f'MST Reconstructed ({split_name} idx={matrix_idx})',
        gics_map=gics_map,
    )

    deg_orig = np.array([deg for _, deg in mst_orig.degree()], dtype=int)
    deg_recon = np.array([deg for _, deg in mst_recon.degree()], dtype=int)
    plot_degree_distribution(deg_orig, deg_recon, sample_dir / 'mst_degree_distribution.png')

    sample_metrics_path = sample_dir / 'mst_comparison_metrics.json'
    pd.DataFrame([metrics]).to_json(sample_metrics_path, orient='records', indent=2)
    print(f'Saved sample metrics: {sample_metrics_path}')

    display(pd.DataFrame([metrics]))
    return metrics


def aggregate_mst_comparison(split_name: str, original_np: np.ndarray, x_tensor: torch.Tensor, max_matrices=None):
    num = original_np.shape[0]
    if max_matrices is not None:
        num = min(num, int(max_matrices))
    if num <= 0:
        return pd.DataFrame()

    recon_all = reconstruct_matrices(model, x_tensor[:num], n_assets=n_assets, batch_size=64)
    rows = []
    for i in range(num):
        orig_corr = sanitize_original_matrix(original_np[i])
        recon_corr = sanitize_reconstruction(recon_all[i])
        _, mst_orig = build_mst_from_corr(orig_corr)
        _, mst_recon = build_mst_from_corr(recon_corr)
        metrics = compare_mst_metrics(mst_orig, mst_recon, n_assets, BETWEENNESS_TOP_K)
        metrics.update({'split': split_name, 'matrix_idx': int(i)})
        rows.append(metrics)

    return pd.DataFrame(rows)


if RUN_SAMPLE_MST:
    run_sample_mst_comparison(MST_SAMPLE_SPLIT, MST_SAMPLE_INDEX)

if RUN_AGGREGATE_MST:
    df_all = aggregate_mst_comparison('all', all_np, x_all, MAX_MATRICES_PER_SPLIT)
    if not df_all.empty:
        all_path = mst_dir / 'mst_comparison_all.json'
        df_all.to_json(all_path, orient='records', indent=2)
        print(f'Saved all metrics: {all_path}')

        metrics_cols = [
            'edge_overlap_pct',
            'edge_jaccard_pct',
            'degree_l1',
            'avg_path_len',
            'avg_path_len_recon',
            'avg_path_len_diff',
            'avg_path_len_unweighted',
            'avg_path_len_unweighted_recon',
            'avg_path_len_unweighted_diff',
            'betweenness_topk_overlap_pct',
        ]
        overall_stats = df_all[metrics_cols].agg(['mean', 'median', 'std', 'min', 'max'])
        summary_row = {'split': 'all'}
        for metric in metrics_cols:
            for stat in ['mean', 'median', 'std', 'min', 'max']:
                summary_row[f'{metric}_{stat}'] = float(overall_stats.loc[stat, metric])
        summary = pd.DataFrame([summary_row])

        summary_path = mst_dir / 'mst_comparison_summary_all.json'
        summary.to_json(summary_path, orient='records', indent=2)
        print(f'Saved summary metrics: {summary_path}')
        display(summary)

In [ ]:
def plot_degree_distribution(deg_orig, deg_recon, out_png):
    max_degree = int(max(deg_orig.max(), deg_recon.max()))
    counts_orig = np.bincount(deg_orig, minlength=max_degree + 1)
    counts_recon = np.bincount(deg_recon, minlength=max_degree + 1)
    freq_orig = counts_orig / counts_orig.sum()
    freq_recon = counts_recon / counts_recon.sum()

    degrees = np.arange(max_degree + 1)
    valid = (freq_orig + freq_recon) > 0
    degrees = degrees[valid]
    freq_orig = freq_orig[valid]
    freq_recon = freq_recon[valid]

    deg_mask = degrees > 0
    degrees = degrees[deg_mask]
    pct_orig = freq_orig[deg_mask] * 100.0
    pct_recon = freq_recon[deg_mask] * 100.0

    value_mask = (pct_orig > 0) | (pct_recon > 0)
    degrees = degrees[value_mask]
    pct_orig = pct_orig[value_mask]
    pct_recon = pct_recon[value_mask]

    if len(degrees) == 0:
        print('No positive degree frequencies to plot in log-log scale.')
        return

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(degrees, pct_orig, s=45, alpha=0.8, color='steelblue', edgecolor='black', label='Original')
    ax.scatter(degrees, pct_recon, s=45, alpha=0.8, color='crimson', edgecolor='black', label='Reconstructed')

    if len(degrees) >= 2:
        slope_orig, intercept_orig = np.polyfit(np.log10(degrees), np.log10(pct_orig), 1)
        y_fit_orig = 10 ** (intercept_orig + slope_orig * np.log10(degrees))
        ax.plot(degrees, y_fit_orig, color='steelblue', linewidth=2, linestyle='--',
                label=f'Original fit slope={slope_orig:.3f}')

        slope_recon, intercept_recon = np.polyfit(np.log10(degrees), np.log10(pct_recon), 1)
        y_fit_recon = 10 ** (intercept_recon + slope_recon * np.log10(degrees))
        ax.plot(degrees, y_fit_recon, color='crimson', linewidth=2, linestyle='--',
                label=f'Reconstructed fit slope={slope_recon:.3f}')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Degree (k)')
    ax.set_ylabel('Nodes with degree k (%)')
    ax.set_title('MST degree distribution (log-log)')
    ax.grid(True, which='both', alpha=0.3)

    xticks = np.unique(degrees.astype(int))
    xticks = xticks[xticks > 0]
    if len(xticks) > 0:
        ax.set_xticks(xticks)
        ax.set_xticklabels([f'{int(k)}' for k in xticks], rotation=45, ha='right')

    yticks = np.unique(np.concatenate([pct_orig, pct_recon]))
    yticks = yticks[yticks > 0]
    if len(yticks) > 0:
        ax.set_yticks(yticks)
        ax.set_yticklabels([f'{val:.4f}%' for val in yticks])

    ax.legend()
    fig.tight_layout()
    fig.savefig(out_png, dpi=160, bbox_inches='tight')
    print(f'Saved degree distribution: {out_png}')
    plt.show()


def run_sample_mst_comparison(split_name: str, matrix_idx: int):
    original_np, x_tensor, meta = get_split_data(split_name)
    if not (0 <= matrix_idx < original_np.shape[0]):
        raise ValueError(f'Invalid matrix index {matrix_idx} for split {split_name}')

    recon_matrix = reconstruct_matrices(model, x_tensor[matrix_idx:matrix_idx + 1], n_assets=n_assets, batch_size=1)[0]
    orig_corr = sanitize_original_matrix(original_np[matrix_idx])
    recon_corr = sanitize_reconstruction(recon_matrix)

    _, mst_orig = build_mst_from_corr(orig_corr)
    _, mst_recon = build_mst_from_corr(recon_corr)

    metrics = compare_mst_metrics(mst_orig, mst_recon, n_assets, BETWEENNESS_TOP_K)
    metrics.update({'split': split_name, 'matrix_idx': int(matrix_idx)})

    sample_dir = mst_dir / f'sample_{split_name}_{matrix_idx}'
    sample_dir.mkdir(parents=True, exist_ok=True)

    tickers = get_tickers(meta, n_assets)
    gics_map = load_gics_map()

    plot_mst_graph(
        mst_orig,
        tickers,
        out_png=sample_dir / 'mst_original_networkx.png',
        out_html=sample_dir / 'mst_original_pyvis.html',
        title=f'MST Original ({split_name} idx={matrix_idx})',
        gics_map=gics_map,
    )
    plot_mst_graph(
        mst_recon,
        tickers,
        out_png=sample_dir / 'mst_reconstructed_networkx.png',
        out_html=sample_dir / 'mst_reconstructed_pyvis.html',
        title=f'MST Reconstructed ({split_name} idx={matrix_idx})',
        gics_map=gics_map,
    )

    deg_orig = np.array([deg for _, deg in mst_orig.degree()], dtype=int)
    deg_recon = np.array([deg for _, deg in mst_recon.degree()], dtype=int)
    plot_degree_distribution(deg_orig, deg_recon, sample_dir / 'mst_degree_distribution.png')

    sample_metrics_path = sample_dir / 'mst_comparison_metrics.json'
    pd.DataFrame([metrics]).to_json(sample_metrics_path, orient='records', indent=2)
    print(f'Saved sample metrics: {sample_metrics_path}')

    display(pd.DataFrame([metrics]))
    return metrics


def aggregate_mst_comparison(split_name: str, original_np: np.ndarray, x_tensor: torch.Tensor, max_matrices=None):
    num = original_np.shape[0]
    if max_matrices is not None:
        num = min(num, int(max_matrices))
    if num <= 0:
        return pd.DataFrame()

    recon_all = reconstruct_matrices(model, x_tensor[:num], n_assets=n_assets, batch_size=64)
    rows = []
    for i in range(num):
        orig_corr = sanitize_original_matrix(original_np[i])
        recon_corr = sanitize_reconstruction(recon_all[i])
        _, mst_orig = build_mst_from_corr(orig_corr)
        _, mst_recon = build_mst_from_corr(recon_corr)
        metrics = compare_mst_metrics(mst_orig, mst_recon, n_assets, BETWEENNESS_TOP_K)
        metrics.update({'split': split_name, 'matrix_idx': int(i)})
        rows.append(metrics)

    return pd.DataFrame(rows)


if RUN_SAMPLE_MST:
    run_sample_mst_comparison(MST_SAMPLE_SPLIT, MST_SAMPLE_INDEX)

if RUN_AGGREGATE_MST:
    df_all = aggregate_mst_comparison('all', all_np, x_all, MAX_MATRICES_PER_SPLIT)
    if not df_all.empty:
        all_path = mst_dir / 'mst_comparison_all.json'
        df_all.to_json(all_path, orient='records', indent=2)
        print(f'Saved all metrics: {all_path}')

        metrics_cols = [
            'edge_overlap_pct',
            'edge_jaccard_pct',
            'degree_l1',
            'avg_path_len',
            'avg_path_len_recon',
            'avg_path_len_diff',
            'avg_path_len_unweighted',
            'avg_path_len_unweighted_recon',
            'avg_path_len_unweighted_diff',
            'betweenness_topk_overlap_pct',
        ]
        overall_stats = df_all[metrics_cols].agg(['mean', 'median', 'std', 'min', 'max'])
        summary_row = {'split': 'all'}
        for metric in metrics_cols:
            for stat in ['mean', 'median', 'std', 'min', 'max']:
                summary_row[f'{metric}_{stat}'] = float(overall_stats.loc[stat, metric])
        summary = pd.DataFrame([summary_row])

        summary_path = mst_dir / 'mst_comparison_summary_all.json'
        summary.to_json(summary_path, orient='records', indent=2)
        print(f'Saved summary metrics: {summary_path}')
        display(summary)